# 01 — Explore one subject
Our milestone is understanding the data, not classifying it. EEG measures voltage differences at scalp electrodes and can contain non-neural artifacts. We use subject 1 and runs 4, 8, 12, chosen before evaluating any outcomes. See the [dataset](https://physionet.org/content/eegmmidb/1.0.0/) and research protocol. Run cells from top to bottom. First execution downloads EDFs; subsequent executions reuse the local cache.

## Load and name channels
MNE reads EDF annotations, standardizes channel names, and attaches template positions. Concatenation retains run boundaries. We preserve the recorded reference and original signals.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
from IPython.display import display
ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise RuntimeError("Launch Jupyter from the repository root or notebooks directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
config = json.loads((ROOT / "experiments/experiment_001/config.json").read_text())
from src.data.load_eeg import load_eeg
from src.features.spectral import compute_psd
from src.visualization.eeg_plots import plot_raw_segment
print("MNE", mne.__version__, "subject", config["subject"], "runs", config["runs"])
raw = load_eeg(config["subject"], config["runs"], ROOT / "data/raw")

In [ ]:
print(raw)
print(raw.info)
print("Channels:", raw.ch_names)
print("Sampling frequency (Hz):", raw.info["sfreq"])
print("Total duration (s):", raw.times[-1])
print("Bad channels:", raw.info["bads"])

## Inspect annotations and explicit event meaning
For these unilateral imagery runs only, T1 means imagined left fist and T2 imagined right fist. T0 is rest. Event IDs generated automatically by MNE are identifiers, not scientific meanings. We assign left=1/right=2 explicitly. BAD/EDGE boundaries mark run joins.

In [ ]:
display(raw.annotations.to_data_frame().head(20))
print("Annotation counts:", pd.Series(raw.annotations.description).value_counts())
from src.preprocessing.epochs import extract_imagery_events
events, event_id = extract_imagery_events(raw, config["runs"])
print("Explicit labels:", event_id)
print("Events [sample, previous value, code]:\n", events[:10])
mne.viz.plot_events(events, sfreq=raw.info["sfreq"], first_samp=raw.first_samp,
                    event_id=event_id, show=False)
plt.show()

## Inspect a short raw segment
C3, Cz and C4 are central scalp channels relevant to our later regional hypothesis. Plotting them does not establish that their signals are artifact-free or more informative.

In [ ]:
plot_raw_segment(raw, start=10., duration=5.)
plt.show()

## Power spectral density
Welch PSD averages windowed spectra to describe power per Hz. The unfiltered plot can reveal broad spectral structure. Mu/alpha and beta are planned analysis bands; peaks alone do not prove motor imagery effects. MNE displays density in dB referenced to µV²/Hz.

In [ ]:
spectrum = compute_psd(raw, fmin=1., fmax=60.)
spectrum.plot(average=True, show=False)
plt.show()

## Check task-window assumptions
The next notebook uses 1–4 seconds after cue onset. Check all task annotation durations, including the minimum, before accepting this interval.

In [ ]:
task = np.isin(raw.annotations.description, ["T1", "T2"])
durations = raw.annotations.duration[task]
print("Task duration min/max (s):", durations.min(), durations.max())
assert durations.min() >= config["tmax"], "Review epoch window: task shorter than tmax."

## Record observations
Record channel count, sampling rate, label counts, task durations and visible artifacts in research/research_log.md. These are inspection observations, not evidence for above-chance decoding. Continue to notebook 02.